# Narsil Vector Embedding Generator

Generates 1536-dim embeddings for TMDB movie data using `gte-Qwen2-1.5B-instruct`.

**Before running:**
1. Go to **Runtime > Change runtime type** and select **T4 GPU**
2. Run all cells (it will prompt you to upload the JSON file)
3. Wait ~10-15 minutes
4. Files auto-download when done

In [ ]:
!pip install -q sentence-transformers umap-learn

In [ ]:
import json
import struct
import time
import os

import numpy as np
import torch
import transformers

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU detected. Go to Runtime > Change runtime type and select T4 GPU.")

In [ ]:
INPUT_FILE = "movies-10000.json"

if not os.path.exists(INPUT_FILE):
    from google.colab import files
    print("Upload movies-10000.json (5.7 MB):")
    uploaded = files.upload()
    INPUT_FILE = list(uploaded.keys())[0]

with open(INPUT_FILE) as f:
    movies = json.load(f)

print(f"Loaded {len(movies)} movies")

texts = []
for m in movies:
    parts = [m.get("title", "")]
    tagline = m.get("tagline", "")
    if tagline:
        parts.append(tagline)
    parts.append(m.get("overview", ""))
    texts.append(". ".join(parts))

print(f"Prepared {len(texts)} text inputs")
print(f"Sample: {texts[0][:200]}...")

In [ ]:
from sentence_transformers import SentenceTransformer

t0 = time.time()
try:
    model = SentenceTransformer(
        "Alibaba-NLP/gte-Qwen2-1.5B-instruct",
        trust_remote_code=False,
        device="cuda",
    )
except Exception as e:
    print(f"Loading without remote code failed ({e}), retrying with trust_remote_code=True...")
    model = SentenceTransformer(
        "Alibaba-NLP/gte-Qwen2-1.5B-instruct",
        trust_remote_code=True,
        device="cuda",
    )
print(f"Model loaded in {time.time() - t0:.1f}s")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
print(f"Generating embeddings for {len(texts)} documents...")
t0 = time.time()
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
elapsed = time.time() - t0
print(f"{len(texts)} embeddings generated in {elapsed:.1f}s ({len(texts)/elapsed:.0f} docs/sec)")
print(f"Shape: {embeddings.shape}")
print(f"L2 norm of first vector: {np.linalg.norm(embeddings[0]):.4f}")

In [ ]:
import umap

print("Running UMAP (1536 -> 3D)...")
t0 = time.time()
reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
coords_3d = reducer.fit_transform(embeddings)
print(f"UMAP completed in {time.time() - t0:.1f}s")
print(f"3D shape: {coords_3d.shape}")

In [ ]:
stem = INPUT_FILE.replace(".json", "")

vectors_path = f"{stem}-vectors-1536.bin"
with open(vectors_path, "wb") as f:
    f.write(struct.pack("<II", len(movies), 1536))
    for vec in embeddings:
        f.write(struct.pack(f"<{1536}f", *vec.tolist()))
print(f"Vectors: {vectors_path} ({os.path.getsize(vectors_path) / 1024 / 1024:.1f} MB)")

coords_path = f"{stem}-umap-3d.bin"
with open(coords_path, "wb") as f:
    f.write(struct.pack("<II", len(movies), 3))
    for coord in coords_3d:
        f.write(struct.pack("<3f", *coord.tolist()))
print(f"UMAP 3D: {coords_path} ({os.path.getsize(coords_path) / 1024 / 1024:.1f} MB)")

ids_path = f"{stem}-vector-ids.json"
with open(ids_path, "w") as f:
    json.dump([m["id"] for m in movies], f)
print(f"IDs: {ids_path}")

print("\nVerification:")
with open(vectors_path, "rb") as f:
    n, d = struct.unpack("<II", f.read(8))
    first_vec = struct.unpack(f"<{d}f", f.read(d * 4))
    norm = sum(x * x for x in first_vec) ** 0.5
    print(f"  {n} vectors x {d} dims, first L2 norm: {norm:.4f}")

In [ ]:
from google.colab import files

print("Downloading output files...")
files.download(vectors_path)
files.download(coords_path)
files.download(ids_path)
print("Done. Move the downloaded files to data/processed/tmdb/ in your Narsil repo.")